# الفصل 50: معماريات الوكلاء والأنظمة متعددة الوكلاء

> وكيل واحد بسيط يكفي للمهام المباشرة، لكن المهام المعقدة (تطوير برمجيات كاملة، بحث متعمّق متعدد الخطوات، أتمتة سير عمل تجاري) تتطلب تخطيطًا هرميًا وتعاونًا بين وكلاء متخصصين — هذا هو الاتجاه الأحدث في أنظمة AI...

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 50
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 49: أسس الوكلاء الذكية: ReAct والأدوات
- **الوسوم:** Multi-Agent، Planning، Orchestration، Communication

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch50-multi-agent-systems.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> وكيل واحد بسيط يكفي للمهام المباشرة، لكن المهام المعقدة (تطوير برمجيات كاملة، بحث متعمّق متعدد الخطوات، أتمتة سير عمل تجاري) تتطلب تخطيطًا هرميًا وتعاونًا بين وكلاء متخصصين — هذا هو الاتجاه الأحدث في أنظمة AI...

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 49: أسس الوكلاء الذكية: ReAct والأدوات


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم التخطيط الهرمي (Hierarchical Planning) — Plan-and-Execute
- فهم أنظمة الوكلاء المتعددة (Multi-Agent Systems)
- فهم Multi-Agent


## 1. لماذا يهمّك هذا الموضوع؟

وكيل واحد بسيط يكفي للمهام المباشرة، لكن المهام المعقدة (تطوير برمجيات كاملة، بحث متعمّق متعدد الخطوات، أتمتة سير عمل تجاري) تتطلب **تخطيطًا هرميًا** و**تعاونًا بين وكلاء متخصصين** — هذا هو الاتجاه الأحدث في أنظمة AI الإنتاجية (مثل Devin، AutoGen، CrewAI).


## 2. الخريطة الذهنية


```text
أنظمة الوكلاء
├── وكيل واحد
│   ├── ReAct: تفكير-فعل-ملاحظة
│   └── Plan-and-Execute: خطة ثم تنفيذ
├── عدة وكلاء
│   ├── Sequential: تسليم متتابع
│   ├── Manager-Worker: مدير يفوّض ويجمع
│   ├── Debate: نقاش وتصحيح متبادل
│   └── Blackboard: مساحة عمل مشتركة
├── البنية التحتية
│   ├── حالة مشتركة (Shared State)
│   ├── تمرير الرسائل (Message Passing)
│   └── نقاط تفتيش بشرية (Human Checkpoints)
└── الاقتصاد
    ├── تكلفة = وكلاء × جولات × tokens
    ├── زمن الاستجابة
    └── قيمة الجودة مقابل التكلفة
```


## 3. الشرح المتدرج

### 🟢 أ) التخطيط الهرمي (Hierarchical Planning) — Plan-and-Execute

بدلاً من التفكير خطوة بخطوة بدون رؤية شاملة (مثل ReAct)، نفصل التخطيط عن التنفيذ:


In [ ]:
class PlanAndExecuteAgent:
    """
    نمط أكثر كفاءة للمهام المعقدة:
    1. مخطِّط (Planner) يُنتج خطة كاملة من البداية
    2. مُنفِّذ (Executor) يُنفّذ كل خطوة من الخطة مستخدمًا ReAct/أدوات
    3. عند الفشل أو ظهور معلومة جديدة، يُعاد التخطيط (Re-planning)
    """
    def __init__(self, planner_llm, executor_agent: ReActAgent):
        self.planner_llm = planner_llm
        self.executor_agent = executor_agent

    def create_plan(self, goal: str) -> list:
        planning_prompt = f"""المهمة: {goal}

قسّم هذه المهمة إلى خطوات فرعية واضحة ومرتبة منطقيًا.
أعطِ النتيجة بصيغة قائمة مرقّمة بسيطة، كل خطوة قابلة للتنفيذ المستقل."""
        response = self.planner_llm.generate(planning_prompt)
        steps = [line.strip() for line in response.split("\n") if line.strip() and line[0].isdigit()]
        return steps

    def execute_plan(self, goal: str) -> dict:
        plan = self.create_plan(goal)
        results = []

        for i, step in enumerate(plan):
            context = f"الهدف الكلي: {goal}\nالخطوات المنجزة: {results}\nالخطوة الحالية: {step}"
            step_result = self.executor_agent.run(context)
            results.append({"step": step, "result": step_result["answer"]})

            # فحص: هل نحتاج إعادة تخطيط بناءً على نتيجة غير متوقعة؟
            if "فشل" in step_result["answer"] or "خطأ" in step_result["answer"]:
                replanning_prompt = f"""الخطة الأصلية: {plan}
الخطوة {i+1} فشلت: {step_result['answer']}
اقترح خطة مُعدَّلة للخطوات المتبقية."""
                revised_steps = self.planner_llm.generate(replanning_prompt)
                plan = plan[:i+1] + [s.strip() for s in revised_steps.split("\n") if s.strip()]

        return {"plan": plan, "execution_results": results}


> 💡 **نصيحة عملية**: نمط Plan-and-Execute أكثر **كفاءة في استهلاك tokens** من ReAct الخالص للمهام الطويلة، لأنه لا يُعيد "التفكير من الصفر" في كل خطوة — لكنه أقل مرونة عند مواجهة مفاجآت تتطلب تغييرًا جذريًا للخطة الأصلية.

### 🟡 ب) أنظمة الوكلاء المتعددة (Multi-Agent Systems)


In [ ]:
class SpecializedAgent:
    """وكيل متخصص بدور واحد محدّد بوضوح"""
    def __init__(self, role: str, system_prompt: str, llm, tools: list = None):
        self.role = role
        self.system_prompt = system_prompt
        self.llm = llm
        self.tools = tools or []

    def process(self, task: str, context: dict = None) -> str:
        full_prompt = f"{self.system_prompt}\n\nالسياق: {context or ''}\nالمهمة: {task}"
        return self.llm.generate(full_prompt)

class MultiAgentOrchestrator:
    """
    منسّق يدير التعاون بين عدة وكلاء متخصصين
    نمط شائع: Manager-Worker (مدير يُوزّع المهام، عمّال متخصصون ينفّذون)
    """
    def __init__(self, manager_llm):
        self.manager_llm = manager_llm
        self.agents = {}

    def register_agent(self, agent: SpecializedAgent):
        self.agents[agent.role] = agent

    def delegate_task(self, goal: str) -> dict:
        # المدير يُقرّر أي وكيل متخصص يجب تكليفه بأي جزء من المهمة
        roles_description = "\n".join([f"- {role}: {a.system_prompt[:100]}"
                                         for role, a in self.agents.items()])
        delegation_prompt = f"""المهمة الكلية: {goal}
الوكلاء المتاحون:
{roles_description}

حدّد تسلسل الوكلاء المطلوبين ومهمة كل واحد، بصيغة JSON:
[{{"role": "...", "subtask": "..."}}]"""

        delegation_plan = json.loads(self.manager_llm.generate(delegation_prompt))

        results = {}
        accumulated_context = {}
        for step in delegation_plan:
            role, subtask = step["role"], step["subtask"]
            if role in self.agents:
                result = self.agents[role].process(subtask, context=accumulated_context)
                results[role] = result
                accumulated_context[role] = result

        synthesis_prompt = f"""المهمة الأصلية: {goal}
نتائج الوكلاء المتخصصين: {results}

ألّف إجابة نهائية متكاملة ومتماسكة."""
        final_answer = self.manager_llm.generate(synthesis_prompt)
        return {"final_answer": final_answer, "agent_contributions": results}

# مثال تطبيقي: فريق كتابة تقرير بحثي
orchestrator = MultiAgentOrchestrator(manager_llm=some_llm)
orchestrator.register_agent(SpecializedAgent(
    role="باحث", system_prompt="أنت باحث متخصص في جمع المعلومات الدقيقة والموثوقة.",
    llm=some_llm, tools=[web_search_tool]
))
orchestrator.register_agent(SpecializedAgent(
    role="محلل_بيانات", system_prompt="أنت محلل بيانات متخصص في استخراج الأنماط والإحصائيات.",
    llm=some_llm, tools=[python_code_executor_tool]
))
orchestrator.register_agent(SpecializedAgent(
    role="كاتب", system_prompt="أنت كاتب محترف يُنتج نصوصًا واضحة ومنظَّمة بأسلوب احترافي.",
    llm=some_llm
))
orchestrator.register_agent(SpecializedAgent(
    role="مراجع", system_prompt="أنت مراجع دقيق يتحقق من الأخطاء المنطقية والواقعية واللغوية.",
    llm=some_llm
))


### 🔴 ج) طبقة التعمّق: أنماط التواصل بين الوكلاء ومشكلة التكلفة

| نمط التعاون | الوصف | مثال إطار عمل |
|---|---|---|
| **Sequential (تتابعي)** | كل وكيل يُنهي عمله قبل تسليم التالي | خط إنتاج بسيط |
| **Hierarchical (هرمي)** | مدير يُفوِّض ويُجمِّع، عمّال متخصصون منفصلون | CrewAI, AutoGen (Manager mode) |
| **Debate (نقاش)** | وكلاء متعددون يتناقشون نفس المسألة للوصول لإجماع | Multi-Agent Debate (تحسين الاستدلال) |
| **Blackboard (لوحة مشتركة)** | كل الوكلاء يقرؤون/يكتبون في مساحة عمل مشتركة | أنظمة معقدة متعددة التخصصات |


In [ ]:
def multi_agent_debate(question: str, agents: list, rounds=3) -> str:
    """
    نمط النقاش: عدة وكلاء (قد تكون نفس LLM بـ temperature مختلفة)
    يتناقشون إجاباتهم لتحسين الدقة النهائية عبر التصحيح المتبادل
    """
    responses = [agent.process(question) for agent in agents]

    for round_num in range(rounds):
        new_responses = []
        for i, agent in enumerate(agents):
            other_responses = [r for j, r in enumerate(responses) if j != i]
            debate_prompt = f"""السؤال: {question}
إجابتك السابقة: {responses[i]}
إجابات الوكلاء الآخرين: {other_responses}

راجع إجابتك في ضوء وجهات النظر الأخرى. إذا وجدت خطأً في تفكيرك، صحّحه.
أعطِ إجابتك المُحدَّثة:"""
            new_responses.append(agent.process(debate_prompt))
        responses = new_responses

    # الإجماع النهائي: التصويت بالأغلبية أو تجميع عبر وكيل حَكَم
    return responses


> ⚠️ **تحذير تكلفة مهم**: أنظمة multi-agent وNقاش (Debate) تُضاعف استهلاك الـ tokens بشكل كبير (N وكلاء × M جولات = استدعاءات LLM كثيرة جدًا). **لا تستخدمها إلا عندما يُثبِت القياس التجريبي** أن التحسّن في الجودة يُبرر الزيادة الهائلة في التكلفة وزمن الاستجابة — للكثير من المهام البسيطة، وكيل واحد جيد التصميم كافٍ تمامًا وأرخص بأضعاف.


## 4. مثال محلول خطوة بخطوة

**المهمة:** إعداد تقرير بحثي عربي عن «أثر الذكاء الاصطناعي على سوق العمل في الخليج».

**الخطة الهرمية:**

| الخطوة | الوكيل | المهمة | المخرج |
|---|---|---|---|
| 1 | باحث | جمع 8 مصادر موثوقة | قائمة مصادر مع ملخصات |
| 2 | محلل بيانات | استخراج الأرقام والاتجاهات | جدول إحصاءات |
| 3 | كاتب | صياغة التقرير | مسودة 1200 كلمة |
| 4 | مراجع | تدقيق الوقائع واللغة | قائمة ملاحظات |
| 5 | كاتب | تطبيق الملاحظات | النسخة النهائية |

**حساب التكلفة:** لنفترض أن كل وكيل يستهلك 2500 token في المتوسط، وأن هناك جولة مراجعة واحدة. إجمالي الاستهلاك ≈ 5 × 2500 = 12500 token. إذا كان سعر المليون token هو 3 دولارات، فتكلفة التقرير ≈ 0.0375 دولار. أضف جولات النقاش (3 وكلاء × 3 جولات) فترتفع التكلفة إلى نحو 0.1 دولار. السؤال ليس «هل هذا رخيص؟» بل «هل الجودة النهائية تبرر 3× تكلفة وكيل واحد؟».

**القياس:** قارن نسخة الوكيل الواحد بنسخة الفريق على 10 تقارير، باستخدام تقييم أعمى: الدقة، التغطية، الاتساق، واللغة. لا تعتمد على انطباعك.


## 5. نصائح وحيل

> 💡 **ابدأ بوكيل واحد جيد**: معظم المهام لا تحتاج فريقًا. أضف وكيلًا ثانيًا فقط عندما تُثبت القياسات فجوة محددة (مثل ضعف التدقيق اللغوي).

> 💡 **اكتب عقدًا لكل وكيل**: الدور، المدخلات المتوقعة، شكل المخرجات، حدود الصلاحيات، ومعايير الإنهاء. الوكلاء بلا عقود يُنتجون فوضى.

> 💡 **استخدم حالة مشتركة مُهيكَلة**: JSON schema واضحة أفضل من نصوص طويلة متبادلة. كل وكيل يقرأ الحقول التي يحتاجها ويكتب حقولًا محددة.

> 💡 **أضف نقطة تفتيش بشرية للأفعال عالية المخاطر**: النشر، الإرسال، الدفع، الحذف. الوكيل يقترح، والإنسان يعتمد.

> 💡 **راقب التكلفة لكل مهمة لا لكل طلب**: التكلفة الحقيقية تشمل إعادة المحاولات وجولات النقاش. ضع ميزانية قصوى لكل مهمة وأوقف النظام عند تجاوزها.


## 6. أخطاء شائعة ومصائد

> ⚠️ **Multi-agent كخيار افتراضي**: كثير من الفرق تبني 5 وكلاء لمهمة يكفيها وكيل واحد، فتحصل على تعقيد وتكلفة بلا مكسب. ابدأ بسيطًا وقِس.

> ⚠️ **أدوار متداخلة**: «باحث» و«محلل» و«كاتب» قد يفعلون نفس الشيء. التداخل يُنتج تكرارًا وتناقضًا. حدّد لكل دور مخرجًا مختلفًا وقابلًا للتحقق.

> ⚠️ **تسلسل الأخطاء**: خطأ في خطوة البحث ينتقل إلى التحليل ثم الكتابة. أضف تحققًا بين المراحل، وليس فقط في النهاية.

> ⚠️ **إغفال الجمود (Deadlock)**: وكيلان ينتظران بعضهما، أو نقاش لا ينتهي. ضع حدًا للجولات وآلية تحكيم.

> ⚠️ **قياس الجودة بالانطباع**: «يبدو أفضل» ليس مقياسًا. استخدم تقييمًا أعمى، ومجموعة مهام ثابتة، ومقارنة تكلفة/زمن/جودة.


## 7. جدول مقارنة

| الإطار | الفلسفة | الأنسب لـ |
|---|---|---|
| **LangGraph** | رسم بياني من الحالات (State Machine) قابل للتحكّم الدقيق | تدفقات عمل معقدة تحتاج تحكّمًا صارمًا |
| **AutoGen (Microsoft)** | محادثات متعددة الوكلاء قابلة للتخصيص | أنظمة بحثية وتجريبية، نقاش بين وكلاء |
| **CrewAI** | أدوار وفرق عمل (crews) بتجريد عالي المستوى | نمذجة فرق عمل بشرية (باحث، كاتب، مراجع) |
| **OpenAI Swarm/Agents SDK** | تناسق خفيف الوزن بين وكلاء متخصصين | تطبيقات بسيطة-متوسطة التعقيد |


## 8. ملخص أفضل الممارسات

- ابدأ **دائمًا** بوكيل واحد بسيط، ولا تنتقل لنظام متعدد الوكلاء إلا إذا أثبتت التجربة أن مهمة واحدة معقدة جدًا لنموذج/prompt واحد.
- حدّد لكل وكيل متخصص **دورًا واحدًا واضحًا** بدلاً من أدوار متداخلة غامضة.
- راقب تكلفة وزمن استجابة النظام متعدد الوكلاء بعناية شديدة — فهي تتصاعد بسرعة.
- ضع حدًا أقصى صارمًا لعدد التكرارات/الجولات لمنع حلقات لا نهائية بين الوكلاء.

---


## 9. أسئلة مقابلات

**سؤال 1: متى تستحق بنية Multi-Agent التعقيد الإضافي؟**

> **إجابة نموذجية:** عندما تكون المهمة قابلة للتفكيك إلى أدوار متخصصة ذات مدخلات ومخرجات واضحة، وعندما تُظهر القياسات أن وكيلًا واحدًا يفشل في جانب محدد (مثل التدقيق أو التحليل الإحصائي). أضف إلى ذلك وجود قيمة واضحة للتوازي (توفير زمن) أو الحاجة إلى نقاش يقلل الخطأ. إذا كانت المهمة بسيطة أو متسلسلة بطبيعتها، فوكيل واحد جيد أرخص وأسهل صيانة.

**سؤال 2: كيف تمنع انتشار الخطأ بين الوكلاء؟**

> **إجابة نموذجية:** تحقق بين المراحل (Schema validation، فحص مصادر، اختبارات)، وليس فقط في النهاية. كل وكيل يجب أن يصرّح بدرجة ثقته ومصادره، والوكيل التالي يرفض المدخلات منخفضة الثقة. أضف «حالة خطأ» صريحة في الـ schema، ومسار تصعيد بشري. وأخيرًا، سجّل كل تحويل حتى يمكن تتبع مصدر الخطأ.

**سؤال 3: كيف تقيّم نظامًا متعدد الوكلاء؟**

> **إجابة نموذجية:** على ثلاثة مستويات: النتيجة النهائية (دقة/جودة مقابل مجموعة ذهبية)، العملية (عدد الخطوات، التكرار، زمن التنفيذ)، والاقتصاد (تكلفة لكل مهمة ناجحة). أضف تقييمًا لكل وكيل على حدة لمعرفة أين تنشأ الأخطاء، وتقييمًا أعمى للنتيجة النهائية. لا يكفي أن يكون النقاش «ممتعًا»؛ يجب أن يُحسّن مقياسًا محددًا.


## 10. تمرين عملي

**الهدف:** تصميم فريق وكلاء لمهمة «تحليل سوق عربي» بميزانية محددة.

**الخطوات:**
1. حدّد المهمة والمخرج النهائي بدقة (تقرير 800 كلمة + جدول مصادر).
2. صمّم 3 وكلاء فقط: باحث، محلل، مراجع. اكتب لكل وكيل عقدًا (مدخلات/مخرجات/صلاحيات).
3. ارسم مخطط التسلسل، وحدّد نقاط التحقق بين المراحل.
4. احسب الميزانية: عدد الاستدعاءات × متوسط tokens × السعر. ضع سقفًا لكل مهمة.
5. نفّذ نسخة مبسطة (حتى بدوال وهمية) وقِس على 5 مهام.
6. قارنها بوكيل واحد على نفس المهام: الجودة، الزمن، التكلفة.
7. اكتب قرارًا: هل تنتقل إلى multi-agent؟ ولماذا؟

**معيار النجاح:** قرار مبرر بالأرقام، لا بالرأي، مع توثيق حالة فشل واحدة على الأقل.


## 11. مراجع للتعمّق

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. arXiv:2308.08155.
- Hong, S., et al. (2023). *MetaGPT: Meta Programming for A Multi-Agent Collaborative Framework*. ICLR 2024.
- Qian, C., et al. (2023). *ChatDev: Communicative Agents for Software Development*. ACL 2024.
- Du, Y., et al. (2023). *Improving Factuality and Reasoning in Language Models through Multiagent Debate*. ICML 2024.
- CrewAI Documentation. docs.crewai.com.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 50: معماريات الوكلاء والأنظمة متعددة الوكلاء.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
